# Lesson 11.3 - AI Research Methods & AI-for-Science (experiment & paper-reading templates)

This notebook provides a reproducible experiment structure and a paper critique checklist to strengthen AI research rigor.

## Objectives

1. Use a structured experiment protocol.
2. Compare baseline models on a controlled split.
3. Capture ablation-style evidence and reproducibility metadata.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 7
np.random.seed(SEED)

## Experiment Template

### Hypothesis
A tree-based model will improve non-linear discrimination over logistic regression on this dataset.

### Dataset
- Name: Breast Cancer Wisconsin (scikit-learn)
- Task: Binary classification
- Split policy: stratified train/test

### Baselines
1. Logistic Regression
2. Random Forest

### Metrics
- Accuracy
- F1
- ROC-AUC

### Ablations
- Feature subset stress test
- Hyperparameter sensitivity

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

models = {
    "log_reg": LogisticRegression(max_iter=2000, random_state=SEED),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
        }
    )

results_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
results_df

## Simple Ablation: Use Only Top 10 Variance Features

In [ ]:
feature_variance = X_train.var().sort_values(ascending=False)
top10_features = feature_variance.head(10).index.tolist()

X_train_top10 = X_train[top10_features]
X_test_top10 = X_test[top10_features]

ablation_rows = []
for name, model in models.items():
    model.fit(X_train_top10, y_train)
    pred = model.predict(X_test_top10)
    proba = model.predict_proba(X_test_top10)[:, 1]
    ablation_rows.append(
        {
            "model": name,
            "accuracy_top10": accuracy_score(y_test, pred),
            "f1_top10": f1_score(y_test, pred),
            "roc_auc_top10": roc_auc_score(y_test, proba),
        }
    )

ablation_df = pd.DataFrame(ablation_rows)
ablation_df

## Merge Main + Ablation Results

In [ ]:
comparison = results_df.merge(ablation_df, on="model")
comparison

## Paper Review Checklist (Reusable)

1. What exact claim does the paper make?
2. Are baselines strong and fairly tuned?
3. Is data split protocol leak-free?
4. Are metrics suitable for the task and class balance?
5. Are variance/error bars reported across runs?
6. Do ablations isolate key components?
7. Are limitations and failure modes explicit?
8. Are code/data artifacts available for reproduction?
9. Do conclusions exceed observed evidence?
10. What would invalidate the main claim?

## Business / Research Case Studies & Exceptions

### Case 1: Healthcare risk model
- Small metric gains can be meaningless without calibration and subgroup checks.
- Exception: a simpler baseline can be preferred for interpretability and auditability.

### Case 2: AI-for-science candidate ranking
- Use uncertainty-aware ranking before expensive experiments.
- Exception: if simulator mismatch is high, ranking confidence can be misleading.

### Case 3: Benchmark-only improvements
- Improvements may fail in deployment if train/test assumptions differ from production.

## Interview Questions & Answers

1. **What is an ablation study?**  
A controlled removal/change of components to measure each component's contribution.

2. **How do you ensure a fair model comparison?**  
Same split, preprocessing, evaluation metrics, and transparent hyperparameter policy.

3. **Why are strong baselines critical?**  
Weak baselines create fake progress.

4. **What is data leakage?**  
Using information during training that would be unavailable at real inference time.

5. **How do you decide metrics?**  
Based on class balance, task objective, and decision consequences.

6. **Why report variance across seeds?**  
Single-run results can be unstable and misleading.

7. **What makes research reproducible?**  
Versioned data/code, environment details, and exact experiment scripts.

8. **How do you read papers critically?**  
Validate claims against protocol quality, baselines, and limitations.

9. **What is simulation-based inference in plain language?**  
Using simulated worlds to infer hidden parameters explaining observed data.

10. **Why should AI engineers care about research methods?**  
It prevents false confidence and improves production decision quality.

11. **What is external validity?**  
Whether a result generalizes beyond the paper's benchmark setup.